# 02 - Feature engineering, model training and MLflow tracking
Three model families are tuned with 5-fold stratified grid search on 80% of the data. The best one by mean CV ROC-AUC becomes the champion. The 20% hold-out set is only used for the final report numbers.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 140)

## Preprocessing pipeline
Engineered feature `hr_reserve_pct = thalach / (220 - age)` (share of age-predicted max heart rate). Numeric: median impute + standard scale. Binary: most-frequent impute. Categorical (`cp`, `restecg`, `slope`, `thal`): most-frequent impute + one-hot.

In [ ]:
from heartrisk.features import build_preprocessor, split_xy
from heartrisk import data
X, y = split_xy(data.load_clean())
pre = build_preprocessor().fit(X)
names = pre.named_steps['columns'].get_feature_names_out()
print(len(names), 'model features:'); print(list(names))

## Train all candidates (logged to MLflow)

In [ ]:
from heartrisk.train import train, candidates
for k, (est, grid) in candidates().items():
    print(k, {p.replace('model__',''): v for p, v in grid.items()})

In [ ]:
meta = train()

In [ ]:
cmp = pd.read_csv('reports/model_comparison.csv', index_col=0)
cmp[['cv_roc_auc','cv_accuracy','cv_precision','cv_recall','test_roc_auc','test_accuracy','test_precision','test_recall','threshold']]

## Decision threshold
For a screening tool a missed patient (false negative) is worse than a false alarm. The threshold is chosen from out-of-fold training predictions as the highest value that still gives >= 90% recall, then applied unchanged to the hold-out set.

In [ ]:
import json; print(json.dumps(meta, indent=2, default=str))

In [ ]:
m = meta['model_name']
for f in [f'confusion_matrix_{m}.png', f'roc_curve_{m}.png', f'feature_importance_{m}.png']:
    display(Image(f'reports/figures/{f}'))

## Browse the runs
```
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```
then open http://localhost:5000 -> experiment `heart-disease-risk`.

In [ ]:
import mlflow
mlflow.set_tracking_uri(f'sqlite:///{ROOT}/mlflow.db')
runs = mlflow.search_runs(experiment_names=['heart-disease-risk'])
runs[['tags.mlflow.runName','metrics.cv_roc_auc','metrics.test_roc_auc','metrics.test_recall','params.decision_threshold','start_time']].head(10)